<a href="https://colab.research.google.com/github/dhushyanthk/AI-Lab/blob/main/EXP10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", g_cost=0):
        self.board = board  # 3x3 tuple representing the puzzle state
        self.parent = parent  # Previous state
        self.move = move  # Move that led to this state
        self.g_cost = g_cost  # Cost from start node to this node
        self.h_cost = self.calculate_manhattan_distance()  # Heuristic cost
        self.f_cost = self.g_cost + self.h_cost # Total estimated cost

    def __lt__(self, other):
        # For use with heapq (priority queue)
        return self.f_cost < other.f_cost

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

    def calculate_manhattan_distance(self):
        distance = 0
        # Goal state: (1, 2, 3, 4, 5, 6, 7, 8, 0)
        # Or you can define it dynamically if the problem statement implies a different goal.
        # For 8-puzzle, the goal is usually 1-8 in order, then 0.
        goal_positions = {
            1: (0, 0), 2: (0, 1), 3: (0, 2),
            4: (1, 0), 5: (1, 1), 6: (1, 2),
            7: (2, 0), 8: (2, 1), 0: (2, 2)
        }

        for i in range(3):
            for j in range(3):
                tile = self.board[i * 3 + j]
                if tile != 0:
                    current_row, current_col = i, j
                    goal_row, goal_col = goal_positions[tile]
                    distance += abs(current_row - goal_row) + abs(current_col - goal_col)
        return distance

    def get_blank_position(self):
        index = self.board.index(0)
        return index // 3, index % 3

    def get_neighbors(self):
        neighbors = []
        blank_row, blank_col = self.get_blank_position()

        moves = {
            "Up": (-1, 0),
            "Down": (1, 0),
            "Left": (0, -1),
            "Right": (0, 1)
        }

        for move_name, (dr, dc) in moves.items():
            new_row, new_col = blank_row + dr, blank_col + dc

            if 0 <= new_row < 3 and 0 <= new_col < 3:
                # Create new board by swapping blank with the tile at (new_row, new_col)
                new_board_list = list(self.board)
                blank_idx = blank_row * 3 + blank_col
                target_idx = new_row * 3 + new_col

                new_board_list[blank_idx], new_board_list[target_idx] = \
                    new_board_list[target_idx], new_board_list[blank_idx]

                new_board = tuple(new_board_list)
                neighbors.append(PuzzleState(new_board, self, move_name, self.g_cost + 1))
        return neighbors

def print_puzzle(board):
    for i in range(3):
        print("|", end=" ")
        for j in range(3):
            val = board[i * 3 + j]
            if val == 0:
                print(" ", end=" |")
            else:
                print(val, end=" |")
        print()
    print() # Newline for separation

In [5]:
def solve_8_puzzle(initial_board):
    goal_board = (1, 2, 3, 4, 5, 6, 7, 8, 0)
    start_state = PuzzleState(initial_board)

    open_set = []  # Priority queue
    heapq.heappush(open_set, start_state)

    closed_set = set() # Stores visited board states

    while open_set:
        current_state = heapq.heappop(open_set)

        if current_state.board == goal_board:
            path = []
            while current_state:
                path.append(current_state)
                current_state = current_state.parent
            return path[::-1] # Return path from start to goal

        closed_set.add(current_state.board)

        for neighbor in current_state.get_neighbors():
            if neighbor.board not in closed_set:
                # Check if neighbor with higher g_cost is already in open_set
                # This is a simplified check; a full A* would update if a better path is found
                # For 8-puzzle, this simple check often suffices.
                # More robust: check if neighbor is in open_set and current g_cost is better.
                # For simplicity here, we add if not in closed_set.
                heapq.heappush(open_set, neighbor)
    return None # No solution found

In [6]:
# Define an initial puzzle state (use 0 for the blank space)
# This board is solvable
initial_board_solvable = (
    1, 2, 3,
    0, 4, 6,
    7, 5, 8
)

# This board is also solvable
# initial_board_solvable = (
#     2, 8, 3,
#     1, 6, 4,
#     7, 0, 5
# )

# This board is unsolvable (odd number of inversions)
# initial_board_unsolvable = (
#     1, 2, 3,
#     4, 5, 6,
#     8, 7, 0
# )

print("Initial Puzzle:")
print_puzzle(initial_board_solvable)

solution_path = solve_8_puzzle(initial_board_solvable)

if solution_path:
    print(f"Solution found in {len(solution_path) - 1} moves:\n")
    for i, state in enumerate(solution_path):
        if i == 0:
            print("Start State:")
        else:
            print(f"Move {i}: {state.move} (g={state.g_cost}, h={state.h_cost}, f={state.f_cost})")
        print_puzzle(state.board)
else:
    print("No solution found for this puzzle.")

Initial Puzzle:
| 1 |2 |3 |
|   |4 |6 |
| 7 |5 |8 |

Solution found in 3 moves:

Start State:
| 1 |2 |3 |
|   |4 |6 |
| 7 |5 |8 |

Move 1: Right (g=1, h=2, f=3)
| 1 |2 |3 |
| 4 |  |6 |
| 7 |5 |8 |

Move 2: Down (g=2, h=1, f=3)
| 1 |2 |3 |
| 4 |5 |6 |
| 7 |  |8 |

Move 3: Right (g=3, h=0, f=3)
| 1 |2 |3 |
| 4 |5 |6 |
| 7 |8 |  |

